# 93장 · 서비스 상태와 종료 조건

상태 전이표는 입력과 현재 상태에 따라 다음 작업이나 종료 위치가 어떻게 정해지는지 나타냅니다.

질문 검증, 자료 조회, 출처 표시, 종료를 상태별로 나눕니다. Q03의 일정 문의가 no_evidence로 끝나는 조건과 빈 입력의 처리 경로를 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## next_step()의 상태 전이

설계안　질문 수신 → 빈 입력 검사 → 장비나 질문 목적 확인 → 허용 문서 검색 → 답과 출처 표시. 공백 질문이면 invalid_input에서, 장비를 특정하기 어려우면 need_clarification에서 끝냅니다. 문서에 없는 교체 일정은 no_evidence로 마칩니다. 검색 결과가 있을 때만 answered로 원문 위치를 보냅니다. 인증·모델·외부 도구는 이 로컬 규칙을 구현하는 데 필요하지 않습니다.


### 셀 01 · next_step()의 종료 상태

정해 둔 일정 질문을 넣었을 때 자료를 읽는 단계까지 가는지 예측하세요.

no_evidence는 여기서 사용자에게 근거 부족을 안내하고 끝낼 수 있는 상태입니다. check_equipment는 다음 함수로 보낼 내부 단계 이름입니다. 화면에 check_equipment라는 문자열을 최종 답으로 표시하지 않습니다.


In [1]:
def next_step(question):
    if not question.strip():
        return "invalid_input"
    if "교체" in question or "일정" in question:
        return "no_evidence"
    return "check_equipment"

print(next_step("장비 A를 다음 달에 교체하기로 확정했나요?"))

no_evidence


no_evidence에서 멈춥니다. 이 분기는 가상 규정에 일정이 없다는 현재 전제를 코드에 옮긴 것입니다. 일정표가 생기면 분기 자체를 다시 검토해야 합니다.

이 예제에서는 공개 규정에 교체 일정이 없다고 정했으므로, 일정을 묻는 질문에는 검색을 실행하기 전에 no_evidence를 반환합니다. 빈 문자열을 이 경로로 보내면 사용자는 질문을 입력해야 할 때 자료가 없다는 안내를 받습니다. 입력 검사를 검색보다 먼저 두는 이유입니다.


#### 검색으로 넘어가기 전의 종료 상태

<mark>빈 입력과 교체 일정 문의는 서로 다른 조건에서 멈춥니다.</mark> next_step에 공백만 넣은 경우와 일정을 묻는 문장을 넣은 경우를 나눠 보세요. 상태 이름을 분리해 두면 화면도 입력 보완과 근거 부재를 따로 안내할 수 있습니다.

선수 개념 · 87장 사용자 시나리오와 추가 질문 조건

#### next_step의 조건 순서

```mermaid
flowchart TD
q["질문"] --> v{"공백을 빼면 비는가?"}
v -->|예| i["invalid_input"]
v -->|아니오| c{"교체 또는 일정 단서?"}
c -->|있음| n["no_evidence"]
c -->|없음| e["check_equipment"]
```

strip 결과가 비었는지 먼저 검사합니다. 그다음 교체·일정 단서를 보고 자료 밖 질문을 분리한 뒤 장비 확인 단계로 넘깁니다.

## 답변·출처 필드와 화면 상태 연결

응답 설계 예시　입력 Q01 / 검색어 장비 A / 후보 RULE-A / 본문 점검 전 전원 표시등을 확인합니다. / 표시할 위치 가상 장비실 규정 A, 1쪽 2.1절, 2026-11-01 / 문서판 rules-v1. 저장을 붙이는 팀은 답을 받는 단계와 저장 성공 여부를 분리하세요. 저장 실패 때문에 근거 없는 질문으로 보이면 사용자 행동이 달라집니다.


### 과제 02 · 선택 연습 · 빈 입력의 상태 전이

공백만 넣었을 때 짧은 분기의 종료 위치를 표시하세요. 질문을 더 받으면 되는 상태와 구분해 봅니다.


In [2]:
# next_step을 호출하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(next_step(&quot;   &quot;))</code></pre>
<p>실행 결과</p><pre><code>invalid_input</code></pre>
<p>invalid_input을 돌려줍니다. 여기서 멈추면 자료를 읽는 비용도 생기지 않습니다.</p>
</details>


## 라우터가 종료시킨 요청을 노드 방문 기록으로 확인하기

92장에서 Q03은 검색 전에 끝내기로 했습니다. 노트북의 next_step을 실제 LangGraph 조건부 간선에 연결하고 Q03의 방문 노드를 출력합니다. 이 작은 그래프는 경로 설계를 검증하기 위한 것입니다.

### 셀 04 · 조건부 간선의 두 목적지 등록

분류 결과가 check_equipment일 때에만 lookup으로 갑니다. 나머지 경로에서는 classify가 적어 둔 판단을 보존하고 종료합니다.

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class RouteState(TypedDict):
    question: str
    route: str
    visits: list[str]

def classify(state):
    return {"route": next_step(state["question"]), "visits": ["classify"]}

def mark_lookup(state):
    return {"visits": state["visits"] + ["lookup"]}

route_graph = StateGraph(RouteState)
route_graph.add_node("classify", classify)
route_graph.add_node("lookup", mark_lookup)
route_graph.add_edge(START, "classify")
route_graph.add_conditional_edges("classify", lambda s: s["route"],
    {"invalid_input": END, "no_evidence": END, "check_equipment": "lookup"})
route_graph.add_edge("lookup", END)
routes = route_graph.compile()
routed = routes.invoke({"question": "장비 A의 교체 일정은?", "route": "", "visits": []})
print(routed["route"], routed["visits"])

no_evidence ['classify']


no_evidence 뒤에는 classify만 남습니다. lookup은 방문할 때만 기록을 더하므로 Q03에서 검색 노드를 건너뛰었다는 것을 확인할 수 있습니다. mark_lookup은 검색 구현이 들어갈 자리입니다.

### 팀 그래프에서 경로를 추적하기

질문을 “장비 A의 점검 전 확인 사항은?”으로 바꿉니다. 이번에는 lookup까지 기록돼야 합니다. 장비명이 빠진 점검 질문도 이 간단한 분류기에서는 check_equipment가 나옵니다. 팀의 조회 노드에서 장비를 정할 수 없을 때 어디서 되묻고 끝낼지 표시하세요.

<mark>끝났다는 출력만으로는 도구 호출을 생략했는지 알 수 없습니다.</mark> 방문 노드 목록에 lookup이 있으면 일단 조회 단계에 진입했습니다. 실제 조회 건수는 조회 함수 안에서 따로 확인합니다.

94장에서 조회 노드가 읽을 원문·판본·권한 정보를 준비하고 95장에서 실제 검색 함수로 연결합니다.

## 돌아보기

94장에는 검색기가 받아야 할 필드와 응답에 남길 필드를 적어 넘깁니다. 화면·저장 방식이 달라도 문서 ID와 버전으로 같은 근거를 가리킬 수 있어야 합니다.
